<a href="https://colab.research.google.com/github/HaninMaatallah/industrial-predictive-maintenance/blob/main/01_eda_and_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# data cleaning

from google.colab import files

uploaded = files.upload()

import io

filename = next(iter(uploaded))

df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nFailure distribution:")
print(df["Machine failure"].value_counts())


df_clean = df.copy()


df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)


duplicate_count = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().copy()


print("Duplicates removed:", duplicate_count)
print("Missing values by column:")
print(df_clean.isna().sum())


sensor_columns = [
    "air_temperature_k",
    "process_temperature_k",
    "rotational_speed_rpm",
    "torque_nm",
    "tool_wear_min"
]

for col in sensor_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("\nData types after validation:")
print(df_clean[sensor_columns].dtypes)

print("\nFinal cleaned shape:", df_clean.shape)

df_clean.to_csv("ai4i2020_clean.csv", index=False)

print("Cleaned dataset exported!")

files.download("ai4i2020_clean.csv")

# Data Analysis

failure_counts = df_clean["machine_failure"].value_counts()

print(failure_counts)

failure_rate = df_clean["machine_failure"].mean() * 100
print(f"\nFailure rate: {failure_rate:.2f}%")

sns.countplot(data=df_clean, x="machine_failure")
plt.title("Machine Failure Distribution")
plt.xlabel("Machine failure (0 = No, 1 = Yes)")
plt.ylabel("Number of records")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(
    data=df_clean,
    x="rotational_speed_rpm",
    kde=True,
    ax=axes[0]
)
axes[0].set_title("Rotational Speed Distribution")

sns.histplot(
    data=df_clean,
    x="tool_wear_min",
    kde=True,
    ax=axes[1]
)
axes[1].set_title("Tool Wear Distribution")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(
    data=df_clean,
    x="machine_failure",
    y="torque_nm",
    ax=axes[0]
)
axes[0].set_title("Torque vs Machine Failure")

sns.boxplot(
    data=df_clean,
    x="machine_failure",
    y="process_temperature_k",
    ax=axes[1]
)
axes[1].set_title("Process Temperature vs Machine Failure")

plt.tight_layout()
plt.show()

numeric_df = df_clean.select_dtypes(include="number")

plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


# Data Preparing
df_model = df_clean.copy()


df_model["temperature_difference_k"] = (
    df_model["process_temperature_k"]
    - df_model["air_temperature_k"]
)


df_model["estimated_power_w"] = (
    df_model["torque_nm"]
    * df_model["rotational_speed_rpm"]
    * (2 * np.pi / 60)
)

print(df_model[
    [
        "temperature_difference_k",
        "estimated_power_w"
    ].head()
])


y = df_model["machine_failure"]


columns_to_drop = [
    "machine_failure",
    "udi",
    "product_id",
    "twf",
    "hdf",
    "pwf",
    "osf",
    "rnf"
]

X = df_model.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("Feature columns:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTraining failure rate:", round(y_train.mean(), 4))
print("Testing failure rate:", round(y_test.mean(), 4))

# Building Random Forest model

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

print("Random Forest model trained successfully!")


# Evaluation of the model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("Average precision:", round(average_precision_score(y_test, y_prob), 4))

print("\nDetailed classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No Failure", "Failure"],
    cmap="Blues"
)

plt.title("Random Forest Confusion Matrix")
plt.show()

feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = model.named_steps[
    "classifier"
].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(importance_df.head(15))

top_features = importance_df.head(15).sort_values("importance")

plt.figure(figsize=(10, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.title("Top 15 Random Forest Features")
plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()

# Generating alerts

ALERT_THRESHOLD = 0.30

maintenance_report = X_test.copy()

maintenance_report["failure_probability"] = y_prob
maintenance_report["actual_failure"] = y_test.to_numpy()

maintenance_report["alert"] = np.where(
    maintenance_report["failure_probability"] >= ALERT_THRESHOLD,
    "INSPECT",
    "NORMAL"
)

maintenance_report = maintenance_report.sort_values(
    "failure_probability",
    ascending=False
)

display(maintenance_report.head(20))

maintenance_report.to_csv(
    "maintenance_alerts.csv",
    index=True,
    index_label="record_index"
)

print("Maintenance report generated!")

files.download("maintenance_alerts.csv")

import joblib

joblib.dump(model, "random_forest_maintenance.joblib")

print("Trained model saved!")


Saving train indus.csv to train indus (3).csv
Dataset loaded successfully!
Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


Dataset dimensions: (10000, 14)

Column names:
['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Data types:
UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

Missing values:
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>